In [5]:
import pandas as pd
import numpy as np
import folium
import geopy

In [6]:
base_df = pd.read_csv("../data/base_df.csv")
print(base_df.shape)

(467, 13)


In [7]:
def get_color(risk_score):
    return "green" if risk_score <= 25 else "orange" if risk_score > 25 and risk_score <= 50 else "red" if risk_score > 50 and risk_score <= 75 else "darkred"

df = pd.read_csv(r"C:\Users\Shiva\Projects\global-aqi-risk\data\base_df.csv")
world_map = folium.Map(location=[20,2],zoom_start=2)

In [8]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="global-aqi-risk")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

coords = []

for _, row in base_df.iterrows():
    query = f"{row['City/Town']}, {row['Country']}"
    location = geocode(query)
    if location:
        coords.append({"City/Town": row["City/Town"], 
                       "Country": row["Country"],
                       "latitude": location.latitude, 
                       "longitude": location.longitude})
    else:
        coords.append({"City/Town": row["City/Town"],
                       "Country": row["Country"], 
                       "latitude": None, 
                       "longitude": None})

coords_df = pd.DataFrame(coords)
print(coords_df.shape)
print(coords_df)

KeyboardInterrupt: 

In [9]:
print(coords_df)


NameError: name 'coords_df' is not defined

In [ ]:
print(coords_df.isnull().sum())

City/Town    0
Country      0
latitude     5
longitude    5
dtype: int64


In [ ]:
map_df = pd.merge(base_df, coords_df, on=["City/Town", "Country"], how="inner")
print(map_df.shape)
print(map_df.isnull().sum())

(471, 15)
Country                     0
City/Town                   0
Year                        0
PM2.5                       0
PM10                        0
CO₂ emissions per capita    0
Renewables                  0
pm25_normalized             0
pm10_normalized             0
co2_normalized              0
ren_normalized              0
risk_score                  0
risk_category               0
latitude                    5
longitude                   5
dtype: int64


In [10]:
map_df = map_df.dropna(subset=["latitude", "longitude"])
print(map_df.shape)

NameError: name 'map_df' is not defined

In [ ]:
map_df.to_csv("../data/map_df.csv", index=False)

NameError: name 'map_df' is not defined

In [11]:
def get_color(risk_score):
    return "green" if risk_score <= 25 else "orange" if risk_score > 25 and risk_score <= 50 else "red" if risk_score > 50 and risk_score <= 75 else "darkred"

df = pd.read_csv(r"C:\Users\Shiva\Projects\global-aqi-risk\data\map_df.csv")
world_risk_map = folium.Map(location=[20,0],zoom_start=2)

In [12]:
for _,row in df.iterrows():
    folium.CircleMarker(
        location = [row["latitude"],row["longitude"]],
        radius = 10,
        color = get_color(row["risk_score"]),
        fill = True,
        fill_color = get_color(row["risk_score"]),
        fill_opacity=0.6,
        popup=f"{row['City/Town']} | Risk Score: {row['risk_score']} | PM2.5: {row['PM2.5']}",
        tool_tip="Click me"
    ).add_to(world_risk_map)

In [13]:
world_risk_map

In [16]:
world_risk_map.save("../outputs/world_risk_map.html")